# Auxilliary Amplifier Design

## Imports and Paths

In [2]:
import os
import sys
import subprocess

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sympy as sp

from tabulate import tabulate

sys.path.append(os.path.abspath(".."))
from pyIC.ic_utils import IcUtils as ic
from pyIC.xschem_utils import XschemUtils as xschem

analog_explorer_path = '/home/andreasp/university/projects/analog-explorer'

## AC Response Requirements

The output pole is given by:

$f_{p1} = \frac{1}{2 \pi R_{out} C_L}, \hspace{20pt} R_{out} = r_{o,P} \parallel r_{o,N} $

In [59]:
gain_boost_target = ic.db2gain(120)

i_tail  = 2e-6
gmid    = 20
fp1     = 1e3
cl      = 500e-15

Av = ic.db2gain(60)
Ro = Av / (gmid * i_tail / 2)
gbw = fp1 * Av

aux_pole_offset = fp1 * Av
main_gain_req = gain_boost_target / Av

print(f'Av: {ic.eng_format(Av)}')
print(f'Ro: {ic.eng_format(Ro)}')
print(f'GBW: {ic.eng_format(gbw)}')
print(f'Aux Pole Offset: {ic.eng_format(aux_pole_offset)}')
print(f'Main Gain Req: {ic.eng_format(ic.gain2db(main_gain_req))} dB')

Av: 1.000e3
Ro: 50.000e6
GBW: 1.000e6
Aux Pole Offset: 1.000e6
Main Gain Req: 60.000e0 dB


## Input NMOS Sizing

In [60]:
gmid_1  = gmid
id_1    = i_tail / 2

m1_op = ic.getop(
    analog_explorer_path,
    model='hv_nmos',
    length=10,
    vds=1.6,
    gmid=gmid_1,
    id=id_1*1e9
)

ic.printop(m1_op, title="M1")

╭──────┬────────────╮
│ M1   │            │
├──────┼────────────┤
│ gmid │   19.799e0 │
│ vgs  │ 570.000e-3 │
│ gmro │    1.561e3 │
│ ft   │    1.987e6 │
│ l    │   10.000e0 │
│ w    │   81.441e3 │
│ gm   │  19.799e-6 │
│ ro   │   78.838e6 │
│ id   │   1.000e-6 │
│ cgg  │  1.586e-12 │
╰──────┴────────────╯


## Output NMOS Sizing

In [70]:
gmid_11  = 15
id_11    = i_tail / 2

m11_op = ic.getop(
    analog_explorer_path,
    model='hv_nmos',
    length=10,
    vds=1.6,
    gmid=gmid_11,
    id=id_11*1e9
)

ic.printop(m11_op, title="M11")

╭───────┬─────────────╮
│ M11   │             │
├───────┼─────────────┤
│ gmid  │    14.866e0 │
│ vgs   │  630.000e-3 │
│ gmro  │     1.183e3 │
│ ft    │     3.504e6 │
│ l     │    10.000e0 │
│ w     │    28.998e3 │
│ gm    │   14.866e-6 │
│ ro    │    79.571e6 │
│ id    │    1.000e-6 │
│ cgg   │ 675.128e-15 │
╰───────┴─────────────╯


## Output PMOS Sizing

In [71]:
gmid_9  = 15

m9_op = ic.getop(
    analog_explorer_path,
    model='hv_pmos',
    length=2,
    vds=1.6,
    gmid=gmid_9,
    id=id_11*1e9
)

ic.printop(m9_op, title="M9")

╭──────┬────────────╮
│ M9   │            │
├──────┼────────────┤
│ gmid │   15.096e0 │
│ vgs  │ 700.000e-3 │
│ gmro │    2.954e3 │
│ ft   │   30.631e6 │
│ l    │    2.000e0 │
│ w    │   15.083e3 │
│ gm   │  15.096e-6 │
│ ro   │  195.684e6 │
│ id   │   1.000e-6 │
│ cgg  │ 78.437e-15 │
╰──────┴────────────╯


In [72]:
Ro_true = ic.parallel((m9_op['ro'], m11_op['ro']))
print(f'Ro True: {ic.eng_format(Ro_true)}')

Ro True: 56.569e6
